# Unified N-Dimensional QMOM Notebook (Moment Transport vs Hybrid Transport vs Monte Carlo)

Set `dim` in the **Configuration** cell and run all cells.

| Method | Abbreviation | Description |
|---|---|---|
| Moment Transport | **MT** | Direct ODE evolution of moments via CQMOM quadrature |
| Hybrid Transport | **HT** | Strang splitting: explicit node advection + CQMOM moment update |
| Monte Carlo | **MC** | Stochastic ground-truth particle simulation |

### Import libraries & setup output folder

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import itertools
import os
from matplotlib.ticker import EngFormatter, ScalarFormatter
import tools.flux_calculator as flx
import tools.CQMOM as CQMOM
import tools.ssp_rk_solver as ssp_rk
from tools.initial_NDF import MC_Gaussian_moments

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except:
    plt.style.use("ggplot")

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
    "figure.titlesize": 24,
    "axes.linewidth": 1.5,
    "lines.linewidth": 2.2,
    "lines.markersize": 7,
})

### ⚙️ Configuration — choose dimension here

In [ ]:
# ============================================================
#  USER SETTINGS  –  edit only this cell
# ============================================================

dim = 2            # Dimension: 2, 3, 4, ...

# --- dynamics parameters (dim-specific defaults applied below) ---
time_step     = None   # set to a float to override, else auto
num_particles = None   # set to an int to override, else auto
t_span        = None   # set to (t0, tf) to override, else auto
N_list_MT     = None   # MT node counts,  e.g. [1, 2, 3]
N_list_HT     = None   # HT node counts,  e.g. [1, 2, 3]  (None → same as N_list_MT)
moments_idx   = None   # np.array of moment orders per dim, else auto

# ============================================================
#  Defaults per dimension
# ============================================================
_defaults = {
    1: dict(time_step=1e-2,  num_particles=10_000,  t_span=(0, np.pi),
            N_list_MT=[1, 2, 3],      N_list_HT=[1, 2, 3], moments_idx=np.array([10])),
    2: dict(time_step=1e-2,  num_particles=10_000,  t_span=(0, 2*np.pi),
            N_list_MT=[1, 2, 3],      N_list_HT=[1, 2, 3], moments_idx=np.array([10, 10])),
    3: dict(time_step=1e-1,  num_particles=10_000, t_span=(0, 100),
            N_list_MT=[1, 2, 3],      N_list_HT=[1, 2, 3], moments_idx=np.array([ 6, 6, 6])),
    4: dict(time_step=1e-1,  num_particles=10_000, t_span=(0, 10),
            N_list_MT=[1, 2],         N_list_HT=[1, 2],    moments_idx=np.array([6, 6, 6, 6])),
    5: dict(time_step=1e-1,  num_particles=10_000, t_span=(0, 10),
            N_list_MT=[1, 2],         N_list_HT=[1, 2],    moments_idx=np.array([4, 4, 4, 4, 4])),
    6: dict(time_step=1e-1,  num_particles=10_000, t_span=(0, 10),
            N_list_MT=[1],            N_list_HT=[1],        moments_idx=np.array([2, 2, 2, 2, 2, 2])),
}

assert dim in _defaults, f"dim must be one of {list(_defaults.keys())}"
_d = _defaults[dim]

if time_step     is None: time_step     = _d["time_step"]
if num_particles is None: num_particles = _d["num_particles"]
if t_span        is None: t_span        = _d["t_span"]
if N_list_MT        is None: N_list_MT  = _d["N_list_MT"]
if N_list_HT     is None: N_list_HT     = _d["N_list_HT"]
if moments_idx   is None: moments_idx   = _d["moments_idx"]

# Enforce minimum moment order for both method lists
_N_max = max(max(N_list_MT), max(N_list_HT))
if moments_idx[0] < 2 * _N_max:
    moments_idx[0] = 2 * _N_max

folder_name = f"Media/{dim}D_CQMOM_output"
if dim == 1:
    folder_name = "Media/1D_QMOM_output"
os.makedirs(folder_name, exist_ok=True)

print(f"dim           = {dim}")
print(f"time_step     = {time_step}")
print(f"num_particles = {num_particles}")
print(f"t_span        = {t_span}")
print(f"N_list_MT     = {N_list_MT}")
print(f"N_list_HT     = {N_list_HT}")
print(f"moments_idx   = {moments_idx}")
print(f"output folder = {folder_name}")

### Initial distribution (multivariate Gaussian)

In [ ]:
# ── mean and covariance (dim-specific) ──────────────────────────────────────
_mu_defaults = {
    1: [1],
    2: [0, 1],
    3: [0, 0, 1],
    4: [0, 0, 0, 1],
    5: [0, 0, 0, 0, 1],
    6: [0, 0, 0, 0, 0, 1],

}
mu  = _mu_defaults[dim]
cov = np.eye(dim) * 1e-2

mc_moments, mc_points, mc_weights = MC_Gaussian_moments(
    mu, cov, moments_idx, num_particles
)

initial_moments = np.zeros(moments_idx)
ranges = [range(idx) for idx in moments_idx]
for idx in itertools.product(*ranges):
    initial_moments[idx] = mc_moments[idx]

# ── histogram of marginals ───────────────────────────────────────────────────
_default_colors = ["#E69F00", "#56B4E9", "#009E73", "#CC79A7", "#D55E00"]
_xi_labels      = [rf"$\xi_{{{i+1}}}$" for i in range(dim)]

col_info = [
    {"title": _xi_labels[i], "unit": "m", "color": _default_colors[i % len(_default_colors)]}
    for i in range(dim)
]

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm", "font.size": 20,
    "axes.labelsize": 20, "axes.titlesize": 20,
    "xtick.labelsize": 18, "ytick.labelsize": 18,
    "axes.grid": True, "grid.alpha": 0.5, "grid.linestyle": ":",
})

fig, axs = plt.subplots(1, dim, figsize=(4 * dim, 3), sharey=True, constrained_layout=True)
if dim == 1:
    axs = [axs]

for i, ax in enumerate(axs):
    formatter = EngFormatter(unit=col_info[i]["unit"], useMathText=True)
    data = mc_points[:, i]
    info = col_info[i]
    ax.hist(data, bins=80, density=False,
            color=info["color"], alpha=0.8,
            edgecolor="black", linewidth=0.5,
            histtype="stepfilled", zorder=3)
    ax.set_xlabel(info["title"])
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    if i == 0:
        ax.set_ylabel("Particle count")

plt.savefig(os.path.join(folder_name, f"initial_marginals_{dim}D.png"), dpi=300)
plt.show()

### Particle dynamics

The RHS is dimension-aware. Default dynamics per dimension:
- **2D**: harmonic oscillator  `ẋ = y, ẏ = −x`
- **3D**: rotating dynamics (helical oscillation in the x-direction)
- **4+D**: generic multi-dimensional example `ẋᵢ = xᵢ₊₁ − b·xᵢ`

Feel free to replace `particle_dynamics` with your own system.

In [ ]:
def particle_dynamics(t, nodes_flat):
    nodes = nodes_flat.reshape(dim, -1)
    xi = [nodes[i].copy() for i in range(dim)]

    if dim == 2:
        x, y = xi
        dxdt = 0.5*np.sin(2*np.pi*t)*x + y
        dydt = 0.5*np.sin(2*np.pi*t)*y - x
        derivs = [dxdt, dydt]

    elif dim == 3:
        x, y, z = xi
        b = 0.193
        dxdt =   0.1 * np.sin(0.05*np.pi*t) + 0 * x   
        dydt =   b * z
        dzdt = - b * y
        derivs = [dxdt, dydt, dzdt]

    else:
        b = 0.193
        derivs = [xi[(i + 1) % dim] - b * xi[i] for i in range(dim)]

    return np.hstack(derivs)


def solve_dynamics(t_eval):
    initial_flat = np.hstack([mc_points[:, i] for i in range(dim)]).flatten()
    sol = solve_ivp(
        particle_dynamics, t_span, initial_flat,
        t_eval=t_eval, method="DOP853", vectorized=True
    )
    return sol

### Method functions (MT & HT shared utilities)

In [ ]:
def function(t, xi, k):
    """Transport term for the moment equation: sum_d k_d * xi_d^{k_d-1} * dxi_d/dt * prod_{j!=d} xi_j^k_j."""    
    derivs = particle_dynamics(t, np.vstack(xi)).reshape(dim, -1)
    result = 0
    for d in range(dim):
        if k[d] == 0:
            continue
        term = k[d] * (xi[d] ** (k[d] - 1)) * derivs[d]
        for j in range(dim):
            if j != d:
                term = term * (xi[j] ** k[j])
        result = result + term
    return result


def compute_flux(t, moments, N):
    """Compute the moment transport flux via CQMOM quadrature."""   
    moments = moments[: 2 ** dim * np.prod(moments_idx)].reshape(moments_idx)
    weights, nodes = CQMOM.CQMOM(
        N, moments, adaptive=True,
        rmin=[1e-3] * dim, eabs=[9e-1] * dim, cutoff=1e-10
    )
    flux = np.zeros(moments_idx)
    for idx in itertools.product(*[range(s) for s in moments_idx]):
        flux[idx] += flx.zeroth_order_local_flux(t, nodes, weights, idx, function).item()
    return flux.flatten()


def update_moments(initial_moments, N, t, dt=time_step, adaptive=False):
    """Advance moments one time-step with SSP-RK3."""    
    initial_moments_flat = initial_moments.flatten()
    updated_flat, ts_error = ssp_rk.SSP_RK3(
        state=initial_moments_flat,
        time_step=dt,
        t=t,
        momidx=N,
        compute_flux=compute_flux,
        adaptive=adaptive,
    )
    return updated_flat.reshape(initial_moments.shape), ts_error

In [ ]:
import sys

def progress_bar(progress, width=40):
    filled = int(width * progress) if progress > 0 else 0
    bar = "█" * filled + "-" * (width - filled)
    sys.stdout.write(f"\r|{bar}| {100*progress:6.2f}%")
    sys.stdout.flush()

### Hybrid Transport (HT) helper functions

HT uses **Strang splitting** (2nd-order operator splitting):

1. **Half-step advection** — integrate quadrature nodes forward by $\Delta t/2$ using `particle_dynamics`
2. **Full-step moment update** — run SSP-RK3 on the moment ODE for $\Delta t$
3. **Half-step advection** — integrate nodes forward by another $\Delta t/2$

Moments are recomputed from the transported nodes after each advection sub-step.

In [ ]:
from scipy.integrate import solve_ivp as _solve_ivp

def compute_flux_discontinuous_only(t, moments, N):
    """Compute the moment transport flux via CQMOM quadrature."""   
    moments = moments[: 2 ** dim * np.prod(moments_idx)].reshape(moments_idx)
    weights, nodes = CQMOM.CQMOM(
        N, moments, adaptive=True,
        rmin=[1e-3] * dim, eabs=[1e-10] * dim, cutoff=1e-10
    )
    flux = np.zeros(moments_idx)
    for idx in itertools.product(*[range(s) for s in moments_idx]):
        flux[idx] += 0 # zero for now, add discontinuous part only if desired
    return flux.flatten()

def _ht_advect_half(moments_now, dt_sub, t_current, N_tup):
    """
    Advance moments by advecting the underlying CQMOM nodes for dt_sub.
    Moments are recomputed from the transported node positions.
    The zeroth moment (number density) is conserved by rescaling.
    """

    M0_old = moments_now.flat[0]
    weights, nodes = CQMOM.CQMOM(
        N_tup, moments_now, adaptive=True,
        rmin=[1e-3]*dim, eabs=[1e-10]*dim, cutoff=1e-10
    )

    if np.sum(weights) < 1e-20:
        return moments_now

    # Integrate nodes forward
    try:
        sol_ht = _solve_ivp(
            particle_dynamics,
            [t_current, t_current + dt_sub],
            nodes.flatten(),
            method="RK23",
        )
        transported = sol_ht.y[:, -1].reshape(dim, -1)
    except Exception:
        transported = nodes   # keep stationary on failure

    # Recompute moments from transported nodes
    new_m = np.zeros_like(moments_now)
    for idx in itertools.product(*[range(s) for s in moments_now.shape]):
        monomial = np.ones(transported.shape[1])
        for d_i, p in enumerate(idx):
            monomial = monomial * transported[d_i] ** p
        new_m[idx] = np.dot(weights, monomial)

    # Conserve number density
    M0_new = new_m.flat[0]
    if M0_new > 1e-20:
        new_m *= M0_old / M0_new

    return new_m


def _ht_step(moments_now, t_current, dt, N_tup):
    """
    One full Strang-split HT step: half node advect → full moment update → half node advect.
    Returns (updated_moments, ts_error).
    """
    # 1 – Half-step advection
    m_half = _ht_advect_half(moments_now, dt / 2.0, t_current, N_tup)

    # 2 – Full-step moment transport (advection flux only, via SSP-RK3)
    updated_flat, ts_error = ssp_rk.SSP_RK3(
        state=m_half.flatten(),
        time_step=dt,
        t=t_current + dt / 2.0,
        momidx=list(N_tup),
        compute_flux=compute_flux_discontinuous_only,
        adaptive=True,
    )
    m_updated = updated_flat.reshape(moments_now.shape)
    
    # 3 – Second half-step advection
    m_final = _ht_advect_half(m_updated, dt / 2.0, t_current + dt / 2.0, N_tup)

    return m_final, ts_error

print("HT helper functions defined.")

### Run Hybrid Transport (HT)

In [ ]:
# ===================================================================================================
# Update Nodes and Moments in Strang-Split HT — no CQMOM inside time-stepping loop if pure advection
# ===================================================================================================

from scipy.integrate import solve_ivp as _solve_ivp

def _moments_from_nodes(weights, nodes):
    """
    Exact moment reconstruction by direct summation.
    No inversion — lossless by construction.

    Parameters
    ----------
    weights : (n_nodes,)
    nodes   : (dim, n_nodes)

    Returns
    -------
    moments : array with shape moments_idx
    """
    moms = np.zeros(moments_idx)
    for idx in itertools.product(*[range(s) for s in moments_idx]):
        monomial = np.ones(nodes.shape[1])
        for d, p in enumerate(idx):
            monomial = monomial * nodes[d] ** p
        moms[idx] = np.dot(weights, monomial)
    return moms


def _advect_nodes(weights, nodes, t_start, dt):
    """
    Integrate every quadrature node forward by dt using particle_dynamics.
    Weights are conserved by advection (they encode mass, not position).

    The ODE is solved on the flattened node array so that vectorised
    dynamics (the existing `particle_dynamics` signature) are used directly.

    Parameters
    ----------
    weights : (n_nodes,)       unchanged
    nodes   : (dim, n_nodes)   positions at t_start
    t_start : float
    dt      : float

    Returns
    -------
    nodes_new : (dim, n_nodes)   positions at t_start + dt
    """
    try:
        sol = _solve_ivp(
            particle_dynamics,
            [t_start, t_start + dt],
            nodes.flatten(),          # shape (dim * n_nodes,)
            method="RK23",
            rtol=1e-6, atol=1e-9,
        )
        nodes_new = sol.y[:, -1].reshape(dim, -1)
    except Exception:
        nodes_new = nodes.copy()      # keep stationary on failure

    return nodes_new


def _ht_step_nodes(weights, nodes, t_current, dt):
    """
    One full Strang-split HT macro step carried out entirely in node space.

    Structure
    ---------
    L^adv_{dt/2}  ->  (optional discontinuous source in moment space)  ->  L^adv_{dt/2}

    For purely continuous dynamics (no discontinuous source) the middle step
    is a no-op and CQMOM is NEVER called inside this function.

    Returns
    -------
    weights  : (n_nodes,)        unchanged for pure advection
    nodes    : (dim, n_nodes)    updated node positions at t_current + dt
    moments  : array(moments_idx) reconstructed by direct summation
    """
    # ── Half-step advection ───────────────────────────────────────────────────
    nodes_half = _advect_nodes(weights, nodes, t_current, 0.5 * dt)

    # ── (Optional) discontinuous source — moment space, one inversion here ───
    # Uncomment and fill in if you have a discontinuous source term:
    
    m_half = _moments_from_nodes(weights, nodes_half)
    m_break, _ = ssp_rk.SSP_RK3(
        state=m_half.flatten(), time_step=dt,
        t=t_current + 0.5 * dt, momidx=list(N_tup),
        compute_flux=compute_flux_discontinuous_only, adaptive=False,
    )
    weights, nodes_half = CQMOM.CQMOM(
        N_tup, m_break.reshape(moments_idx),
        adaptive=True, rmin=[1e-3]*dim, eabs=[1e-10]*dim, cutoff=1e-10,
    )

    # ── Second half-step advection ────────────────────────────────────────────
    nodes_new = _advect_nodes(weights, nodes_half, t_current + 0.5 * dt, 0.5 * dt)

    # ── Lossless moment reconstruction ────────────────────────────────────────
    moments_new = _moments_from_nodes(weights, nodes_new)

    return weights, nodes_new, moments_new


print("HT node-tracking helper functions defined.")


# ============================================================
# CELL 2 – Replace the "Run Hybrid Transport (HT)" cell
# ============================================================

print("=" * 60)
print("Running HYBRID TRANSPORT (HT) — node tracking")
print("=" * 60)

HT_Moments = {}   # HT_Moments[N]  shape: (n_frames, *moments_idx)
t_eval_HT  = {}   # t_eval_HT[N]   shape: (n_frames,)

t0_ht, tf_ht = t_span

for N_ht in N_list_HT:
    N_tup = tuple([N_ht] * dim)
    print(f"\nN = {N_ht}")

    # ── Single CQMOM inversion at t = 0 ──────────────────────────────────────
    weights, nodes = CQMOM.CQMOM(
        N_tup, initial_moments,
        adaptive=True,
        rmin=[1e-3]  * dim,
        eabs=[9e-1] * dim,
        cutoff=1e-10,
        rcond=9e-1
    )

    t_ht  = t0_ht
    dt_ht = time_step

    # Record initial frame from node positions (not from initial_moments directly,
    # so the series is self-consistent from the start)
    _frames_m = [_moments_from_nodes(weights, nodes)]
    _frames_t = [t_ht]

    while t_ht < tf_ht:
        dt_ht = min(dt_ht, tf_ht - t_ht)   # don't overshoot final time

        weights, nodes, m_new = _ht_step_nodes(weights, nodes, t_ht, dt_ht)

        t_ht += dt_ht
        _frames_t.append(t_ht)
        _frames_m.append(m_new)

        progress_bar(min((t_ht - t0_ht) / (tf_ht - t0_ht), 1.0))

    progress_bar(1.0)
    print(f"  Done  ({len(_frames_t)} steps)")

    HT_Moments[N_ht] = np.array(_frames_m)   # (n_frames, *moments_idx)
    t_eval_HT[N_ht]  = np.array(_frames_t)

### Run Moment Transport (MT)

In [ ]:
import sys

def progress_bar(progress, width=40):
    filled = int(width * progress) if progress > 0 else 0
    bar = "█" * filled + "-" * (width - filled)
    sys.stdout.write(f"\r|{bar}| {100*progress:6.2f}%")
    sys.stdout.flush()

MT_Moments = {}
t_eval = {}
t0, tf = t_span

for N in N_list_MT:
    N_vec  = [N] * dim  # e.g. [N, N] for 2D, [N, N, N] for 3D, …
    print(f"\nN = {N}")
    current_moments = initial_moments.copy()
    MT_Moments[N] = []
    t_eval[N] = []
    t  = t0
    dt = time_step

    while t + dt < tf:
        t_eval[N].append(t)
        MT_Moments[N].append(current_moments) 
        current_moments, ts_error = update_moments(
            current_moments, [N] * dim, t, dt, adaptive=True
        )
        dt = ssp_rk.adapt_time_step(dt, ts_error, 1e-6, 1e-1, time_step)
        t += dt
        progress_bar(min((t - t0) / (tf - t0), 1.0))

    progress_bar(1.0)
    print("  Done") 
    MT_Moments[N] = np.array(MT_Moments[N])

### Compute MC reference moments

In [ ]:
num_loops = len(t_eval[N_list_MT[0]])
MC_Moments = np.zeros((*moments_idx, num_loops))
sol = solve_dynamics(t_eval[N_list_MT[0]])

for frame in range(num_loops):
    xi = sol.y[:, frame].reshape(dim, -1)
    for idx in itertools.product(*[range(s) for s in moments_idx]):
        val = mc_weights.copy()
        for d, power in enumerate(idx):
            val = val * xi[d] ** power
        MC_Moments[idx + (frame,)] = np.sum(val)

### Plot moment evolution — MT vs HT vs MC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ------------------ STYLE ------------------
plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "cm",
    "font.size": 30, "axes.titlesize": 30, "axes.labelsize": 30,
    "xtick.labelsize": 30, "ytick.labelsize": 30,
    "legend.fontsize": 26, "figure.titlesize": 30,
    "axes.linewidth": 1.5, "lines.linewidth": 2.2, "lines.markersize": 7,
})

logarithmic_scale = False
COLOR_MC = "black"

# ------------------ MOMENTS ------------------
_default_selected = {
    1: [(0,),(1,),(2,),(3,),(4,),(5,)],
    2: [(0,0),(1,1),(2,2),(7,7),(8,8),(9,9)],
    3: [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(0,1,1),(1,1,0)],
    4: [(0,0,0,0),(1,0,0,0),(0,1,0,0),(0,0,1,0),(0,0,0,1),(1,1,0,0)],
    5: [(0,0,0,0,0),(1,0,0,0,0),(0,1,0,0,0),(0,0,1,0,0),(0,0,0,1,0),(0,0,0,0,1)],
    6: [(0,0,0,0,0,0),(1,0,0,0,0,0),(0,1,0,0,0,0),(0,0,1,0,0,0),(0,0,0,1,0,0),(0,0,0,0,1,0)],
}
selected_moments = _default_selected[dim]

# ------------------ COLORS ------------------
cmap_mt   = plt.cm.viridis
colors_mt = [cmap_mt(i / max(len(N_list_MT)-1, 1)) for i in range(len(N_list_MT))]

cmap_ht   = plt.cm.viridis
colors_ht = [cmap_ht(i / max(len(N_list_HT)-1, 1)) for i in range(len(N_list_HT))]

# ------------------ GENERIC PLOT FUNCTION ------------------
def make_figure(data_dict, t_dict, N_values, colors, label_prefix,
                linestyle, filename):

    n_plots = len(selected_moments)
    ncols   = 3
    nrows   = int(np.ceil(n_plots / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 5*nrows))
    axes = axes.flatten()

    # ref_N = N_values[0]
    ref_N = 1

    for plot_idx, idx in enumerate(selected_moments):
        ax = axes[plot_idx]

        fmt_y = ticker.ScalarFormatter(useMathText=True)
        fmt_y.set_scientific(True)
        fmt_y.set_powerlimits((-2, 2))
        fmt_y.set_useOffset(False)
        ax.yaxis.set_major_formatter(fmt_y)

        if all(p == 0 for p in idx):
            ax.set_ylim(0, 2)

        ax.tick_params(width=1.5, length=8, pad=6)
        ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        # MC (reference)
        ax.plot(
            t_eval[ref_N], MC_Moments[idx],
            linestyle="None", marker="s", markersize=8,
            markerfacecolor="none", markeredgecolor=COLOR_MC,
            markeredgewidth=1.8,
            label="MC" if plot_idx == 0 else None,
        )

        # Method curves
        for c_i, N in enumerate(N_values):
            ax.plot(
                t_dict[N], data_dict[N][(slice(None), *idx)],
                linestyle=linestyle, linewidth=2.5,
                marker="o" if linestyle == "-" else None,
                color=colors[c_i],
                label=f"{label_prefix} (N={N})" if plot_idx == 0 else None,
            )

        label_str = ",".join(str(p) for p in idx)
        ax.set_title(rf"$M_{{{label_str}}}$")

        if logarithmic_scale:
            ax.set_yscale("log")

        if plot_idx >= n_plots - ncols:
            ax.set_xlabel(r"Time [s]")

    # remove empty axes
    for k in range(n_plots, len(axes)):
        fig.delaxes(axes[k])

    handles, labels_leg = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_leg, loc="upper center",
               bbox_to_anchor=(0.5, 0.995),
               ncol=min(len(labels_leg), 6),
               frameon=False)

    plt.tight_layout()
    plt.subplots_adjust(top=0.82, hspace=0.4, wspace=0.3)

    plt.savefig(os.path.join(folder_name, filename + ".pdf"),
                bbox_inches="tight")
    plt.show()


# ------------------ CREATE BOTH FIGURES ------------------

# MT figure
make_figure(
    data_dict=MT_Moments,
    t_dict=t_eval,
    N_values=N_list_MT,
    colors=colors_mt,
    label_prefix="MT",
    linestyle="-",
    filename=f"Moments_MT_{dim}D"
)

# HT figure
make_figure(
    data_dict=HT_Moments,
    t_dict=t_eval_HT,
    N_values=N_list_HT,
    colors=colors_ht,
    label_prefix="HT",
    linestyle="-",
    filename=f"Moments_HT_{dim}D"
)

### Phase-space animation (2D only)

In [ ]:
if dim == 2:
    import matplotlib.animation as animation
    from scipy.interpolate import interp1d

    plt.rcParams.update({
        "font.family": "serif", "mathtext.fontset": "cm",
        "font.size": 24, "axes.titlesize": 24, "axes.labelsize": 24,
        "legend.fontsize": 18, "axes.linewidth": 1.5,
    })

    # ── Common time grid (intersection of ALL time arrays) ────────────────────
    time_arrays = ([t_eval[N] for N in N_list_MT]
                   + [t_eval_HT[N] for N in N_list_HT]
                   + [sol.t])
    t_min = max(t[0]  for t in time_arrays)
    t_max = min(t[-1] for t in time_arrays)
    n_anim_frames = 150
    t_anim = np.linspace(t_min, t_max, n_anim_frames)

    # ── Interpolators ─────────────────────────────────────────────────────────
    mt_interp = {
        N: interp1d(t_eval[N], MT_Moments[N], axis=0,
                    kind="linear", fill_value="extrapolate")
        for N in N_list_MT
    }
    ht_interp = {
        N: interp1d(t_eval_HT[N], HT_Moments[N], axis=0,
                    kind="linear", fill_value="extrapolate")
        for N in N_list_HT
    }
    mc_interp = interp1d(sol.t, sol.y, axis=1,
                         kind="linear", fill_value="extrapolate")

    # ── Colours ───────────────────────────────────────────────────────────────
    cmap_mt   = plt.cm.viridis
    temp_mt   = max(len(N_list_MT) - 1, 1)
    colors_mt = [cmap_mt(i / temp_mt) for i in range(len(N_list_MT))]

    cmap_ht   = plt.cm.viridis
    temp_ht   = max(len(N_list_HT) - 1, 1)
    colors_ht = [cmap_ht(i / temp_ht) for i in range(len(N_list_HT))]

    # ── Figure: always 1×2 side-by-side ──────────────────────────────────────
    fig2, (ax_mt, ax_ht) = plt.subplots(1, 2, figsize=(20, 9), sharey=True)
    plt.subplots_adjust(top=0.80, wspace=0.12)

    def _setup_ax(ax, ylabel=True):
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.set_xlabel(r"$\xi_1$")
        if ylabel:
            ax.set_ylabel(r"$\xi_2$")
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    _setup_ax(ax_mt, ylabel=True)
    _setup_ax(ax_ht, ylabel=False)

    # ── Scatter artists — MT panel ────────────────────────────────────────────
    mc_sc_mt = ax_mt.scatter([], [], s=15, color="gray", edgecolors="none",
                              alpha=0.15, label="MC")
    mt_artists = []
    for c_idx, N in enumerate(N_list_MT):
        sc = ax_mt.scatter([], [], s=[40], color=colors_mt[c_idx],
                           edgecolors="black", linewidth=0.8, alpha=0.95,
                           label=f"MT (N={N})")
        mt_artists.append(sc)

    # ── Scatter artists — HT panel ────────────────────────────────────────────
    mc_sc_ht = ax_ht.scatter([], [], s=15, color="gray", edgecolors="none",
                              alpha=0.15, label="MC")
    ht_artists = []
    for c_idx, N in enumerate(N_list_HT):
        sc = ax_ht.scatter([], [], s=[40], color=colors_ht[c_idx],
                           edgecolors="black", linewidth=0.8, alpha=0.95,
                           label=f"HT (N={N})")
        ht_artists.append(sc)

    # ── Legends ───────────────────────────────────────────────────────────────
    h_mt, l_mt = ax_mt.get_legend_handles_labels()
    h_ht, l_ht = ax_ht.get_legend_handles_labels()
    fig2.legend(h_mt + h_ht, l_mt + l_ht,
                loc="upper center", bbox_to_anchor=(0.5, 0.98),
                ncol=len(l_mt) + len(l_ht), frameon=False)

    # ── Animation update ──────────────────────────────────────────────────────
    def update(frame_idx):
        tc = t_anim[frame_idx]

        xi_mc      = mc_interp(tc).reshape(2, -1)
        mc_offsets = np.column_stack([xi_mc[1], xi_mc[0]])
        mc_sc_mt.set_offsets(mc_offsets)
        mc_sc_ht.set_offsets(mc_offsets)

        for c_idx, N in enumerate(N_list_MT):
            m = mt_interp[N](tc).reshape(moments_idx)
            w, xi = CQMOM.CQMOM([N, N], m, adaptive=True,
                                  rmin=[1e-3]*2, eabs=[1e-10]*2, cutoff=1e-10)
            mt_artists[c_idx].set_offsets(np.column_stack([xi[1], xi[0]]))
            mt_artists[c_idx].set_sizes(300 * np.sqrt(w))

        for c_idx, N in enumerate(N_list_HT):
            m = ht_interp[N](tc).reshape(moments_idx)
            w, xi = CQMOM.CQMOM([N, N], m, adaptive=True,
                                  rmin=[1e-3]*2, eabs=[1e-10]*2, cutoff=1e-10)
            ht_artists[c_idx].set_offsets(np.column_stack([xi[1], xi[0]]))
            ht_artists[c_idx].set_sizes(300 * np.sqrt(w))

        title_str = rf"$t = {tc:.2f}\ \mathrm{{s}}$"
        ax_mt.set_title(f"Moment Transport (MT)\n{title_str}", pad=8)
        ax_ht.set_title(f"Hybrid Transport (HT)\n{title_str}", pad=8)

        return [mc_sc_mt] + mt_artists + [mc_sc_ht] + ht_artists

    ani = animation.FuncAnimation(fig2, update,
                                   frames=n_anim_frames, interval=100, blit=False)

    gif_path = os.path.join(folder_name, "PhaseSpace_MTvsHT_2D.gif")
    ani.save(gif_path, writer="pillow", fps=12, dpi=180)
    plt.close(fig2)
    print("Saved animation to:", gif_path)

else:
    print(f"Phase-space animation is only implemented for dim=2 (current dim={dim}).")